# Knowledge Graph for Dietary Recommendations in Disease Management

**Course:** Knowledge Graphs — TU Wien 2026S
**Author:** Serkan Besim

---

> **Medical Disclaimer:** This notebook is an academic course project.
> All dietary suggestions are derived from a simplified model using public datasets
> (USDA FoodData Central, CTD, curated clinical guidelines). They must **not** be
> interpreted as medical advice. Always consult a qualified healthcare professional
> for dietary guidance.

---

This notebook implements a Knowledge Graph (KG) that recommends foods based on diseases.
It integrates three data sources, applies Datalog-style logical reasoning, and trains
RotatE embeddings for link prediction.

| Primary Focus | Description |
|---|---|
| **LO1** | KG Embeddings — RotatE via PyKEEN, link prediction |
| **LO2** | Logical Knowledge — Datalog rules via SPARQL CONSTRUCT |
| **LO7/LO8** | KG construction and rule-based enrichment |
| **LO9/LO11** | Healthcare domain application and recommendation service |
| **LO12** | Neuro-symbolic integration (rules + embeddings) |

**Pipeline overview:**
`Data Loading → Data Preparation → KG Construction → Disease Enrichment → Logical Reasoning → RotatE Training → Recommendation Service`


## 1. Setup
Import required libraries and verify installation.

In [1]:
# ── Standard library ─────────────────────────────────────────────────────────
import re
import json
from collections import Counter
from pathlib import Path

# ── Data ─────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── RDF / Knowledge Graph ────────────────────────────────────────────────────
from rdflib import Graph, Namespace, RDF, RDFS, Literal, URIRef
from rdflib.plugins.sparql import prepareQuery

# ── KG Embeddings ────────────────────────────────────────────────────────────
import torch
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.predict import predict_target

In [2]:
BASE_DIR = Path().resolve()

# Output directory
out_dir = BASE_DIR / "outputs"
out_dir.mkdir(exist_ok=True)

# Input data paths — USDA FoodData Central (foundation food subset)
FOOD_CSV         = BASE_DIR / "food.csv"
NUTRIENT_CSV     = BASE_DIR / "nutrient.csv"
FOOD_NUTRIENT_CSV = BASE_DIR / "food_nutrient.csv"
FOUNDATION_CSV   = BASE_DIR / "foundation_food.csv"

# Input data path — CTD (Comparative Toxicogenomics Database)
CTD_CSV = BASE_DIR / "DiseaseData" / "CTD_curated_chemicals_diseases.csv"

print(f"Working directory : {BASE_DIR}")
print(f"Output directory  : {out_dir}")

Working directory : C:\Users\serka\kg_nutrition
Output directory  : C:\Users\serka\kg_nutrition\outputs


In [3]:
# Key nutrients we care about
key_nutrients = {
    1003: 'Protein',
    1004: 'Fat',
    1005: 'Carbohydrate',
    1079: 'Fiber',
    1087: 'Calcium',
    1089: 'Iron',
    1090: 'Magnesium',
    1092: 'Potassium',
    1093: 'Sodium',
    1095: 'Zinc',
    1162: 'Vitamin_C',
    1114: 'Vitamin_D',
    1175: 'Vitamin_B6',
    1178: 'Vitamin_B12',
    1177: 'Folate',
    1253: 'Cholesterol',
    1063: 'Sugar',
    1258: 'Saturated_Fat',
}

## 2. Data Loading
We use three data sources:
- **USDA FoodData Central** — 365 foundation foods with nutrient compositions
- **Comparative Toxicogenomics Database (CTD)** — curated therapeutic nutrient-disease associations (`DirectEvidence == 'therapeutic'`)
- **Curated dietary guidelines (`worsensWith_curated.csv`)** — nutrient-disease avoidance pairs sourced from NIH ODS, WHO, AHA, ADA, and EASL clinical recommendations


In [4]:
food_df          = pd.read_csv(FOOD_CSV)
food_nutrient_df = pd.read_csv(FOOD_NUTRIENT_CSV, low_memory=False)
foundation_df    = pd.read_csv(FOUNDATION_CSV)

print(f"Total foods in USDA: {len(food_df)}")
print(f"Foundation foods:    {len(foundation_df)}")

Total foods in USDA: 78026
Foundation foods:    365


## 3. Data Preparation
Select 18 clinically relevant nutrients, filter to foundation foods,  
and flag foods as high in a nutrient when their amount exceeds the top quartile threshold.

In [5]:
# Get food names for foundation foods only
foundation_foods = food_df[food_df['fdc_id'].isin(foundation_df['fdc_id'])][['fdc_id', 'description']]

In [6]:
# Join foundation foods with their nutrient values
food_nutrient_filtered = food_nutrient_df[
    (food_nutrient_df['fdc_id'].isin(foundation_foods['fdc_id'])) &
    (food_nutrient_df['nutrient_id'].isin(key_nutrients.keys()))
][['fdc_id', 'nutrient_id', 'amount']]

# Add food names
food_nutrient_filtered = food_nutrient_filtered.merge(
    foundation_foods, on='fdc_id'
)

# Add nutrient names
food_nutrient_filtered['nutrient_name'] = food_nutrient_filtered['nutrient_id'].map(key_nutrients)

# Final clean dataframe
df = food_nutrient_filtered[['fdc_id', 'description', 'nutrient_name', 'amount']].copy()
df.columns = ['fdc_id', 'food', 'nutrient', 'amount']

print(df.shape)
print(df.head(20))

(4189, 4)
    fdc_id                  food       nutrient   amount
0   321358    Hummus, commercial      Magnesium   71.100
1   321358    Hummus, commercial  Saturated_Fat    2.220
2   321358    Hummus, commercial           Iron    2.410
3   321358    Hummus, commercial     Vitamin_B6    0.143
4   321358    Hummus, commercial          Fiber    5.400
5   321358    Hummus, commercial          Sugar    0.340
6   321358    Hummus, commercial        Protein    7.350
7   321358    Hummus, commercial        Calcium   41.000
8   321358    Hummus, commercial      Potassium  289.000
9   321358    Hummus, commercial         Sodium  438.000
10  321358    Hummus, commercial         Folate   36.000
11  321358    Hummus, commercial            Fat   17.100
12  321358    Hummus, commercial           Zinc    1.380
13  321358    Hummus, commercial   Carbohydrate   14.900
14  321358    Hummus, commercial      Vitamin_C    0.000
15  321360  Tomatoes, grape, raw      Vitamin_C   27.200
16  321360  Tomatoes,

In [7]:
# Calculate thresholds per nutrient
thresholds = df.groupby('nutrient')['amount'].quantile([0.25, 0.75]).unstack()
thresholds.columns = ['low_threshold', 'high_threshold']
print(thresholds.round(2))

               low_threshold  high_threshold
nutrient                                    
Calcium                 9.22           72.80
Carbohydrate            2.54           20.14
Cholesterol            53.67           92.00
Fat                     0.32            7.76
Fiber                   1.80            4.45
Folate                  8.99           56.31
Iron                    0.14            2.17
Magnesium              11.90           44.60
Potassium             160.70          375.90
Protein                 1.12           18.19
Saturated_Fat           1.06           11.24
Sodium                  0.59          111.45
Sugar                   2.26            8.88
Vitamin_B12             0.57            1.66
Vitamin_B6              0.06            0.20
Vitamin_C               4.60           34.97
Vitamin_D               0.00            1.15
Zinc                    0.21            2.72


In [8]:
def get_level(row):
    high = thresholds.loc[row['nutrient'], 'high_threshold']
    if row['amount'] >= high:
        return 'high'
    return None

df['level'] = df.apply(get_level, axis=1)
df = df[df['level'] == 'high'].copy()

print(f"Food-nutrient pairs above top quartile threshold: {len(df)}")
print(df.head(5))

Food-nutrient pairs above top quartile threshold: 1055
    fdc_id                food   nutrient  amount level
0   321358  Hummus, commercial  Magnesium   71.10  high
2   321358  Hummus, commercial       Iron    2.41  high
4   321358  Hummus, commercial      Fiber    5.40  high
9   321358  Hummus, commercial     Sodium  438.00  high
11  321358  Hummus, commercial        Fat   17.10  high


## 4. Knowledge Graph Construction
Translate the prepared data into RDF triples using rdflib.  
Each food-nutrient pair becomes a triple, e.g.:  
`(Lentils_dry) -- highIn --> (Iron)`

In [9]:
# Define namespaces
FOOD    = Namespace("http://nutrition-kg.org/food/")
NUTRIENT = Namespace("http://nutrition-kg.org/nutrient/")
PROP    = Namespace("http://nutrition-kg.org/property/")

def clean_uri(name):
    name = name.replace(' ', '_')
    name = re.sub(r'[^a-zA-Z0-9_]', '', name)
    return name

# Build graph
g = Graph()
g.bind("food",     FOOD)
g.bind("nutrient", NUTRIENT)
g.bind("prop",     PROP)

for _, row in df.iterrows():
    food_uri    = FOOD[clean_uri(row['food'])]
    nutrient_uri = NUTRIENT[row['nutrient']]
    g.add((food_uri, PROP['highIn'], nutrient_uri))

print(f"Food-nutrient triples: {len(g)}")
print("\nSample triples:")
for s, p, o in list(g)[:5]:
    print(f"  {s.split('/')[-1]} -- {p.split('/')[-1]} --> {o.split('/')[-1]}")

Food-nutrient triples: 1055

Sample triples:
  Pork_loin_tenderloin_boneless_raw -- highIn --> Potassium
  Seeds_sunflower_seed_kernels_dry_roasted_with_salt_added -- highIn --> Iron
  Sausage_Italian_pork_mild_cooked_panfried -- highIn --> Sodium
  Cheese_oaxaca_solid -- highIn --> Calcium
  Fish_salmon_sockeye_wild_caught_raw -- highIn --> Protein


In [10]:
# Add type information
for _, row in df[['food']].drop_duplicates().iterrows():
    food_uri = FOOD[clean_uri(row['food'])]
    g.add((food_uri, RDF.type, PROP['Food']))

for nutrient_name in key_nutrients.values():
    nutrient_uri = NUTRIENT[nutrient_name]
    g.add((nutrient_uri, RDF.type, PROP['Nutrient']))

print(f"Total triples after adding types: {len(g)}")

Total triples after adding types: 1387


In [11]:
# ── Property labels ──────────────────────────────────────────────────────────
g.add((PROP['highIn'],           RDFS.label, Literal("is high in nutrient")))
g.add((PROP['treats'],           RDFS.label, Literal("therapeutically treats disease")))
g.add((PROP['worsensWith'],      RDFS.label, Literal("worsens disease")))
g.add((PROP['recommendedFor'],   RDFS.label, Literal("recommended for disease (inferred)")))
g.add((PROP['contraindicatedFor'], RDFS.label, Literal("contraindicated for disease (inferred)")))
g.add((PROP['predictedFor'],     RDFS.label, Literal("embedding-predicted recommendation")))

# ── Domain / Range declarations (lightweight RDFS schema) ────────────────────
g.add((PROP['highIn'], RDFS.domain, PROP['Food']))
g.add((PROP['highIn'], RDFS.range,  PROP['Nutrient']))

g.add((PROP['treats'],             RDFS.domain, PROP['Nutrient']))
g.add((PROP['treats'],             RDFS.range,  PROP['Disease']))
g.add((PROP['worsensWith'],        RDFS.domain, PROP['Nutrient']))
g.add((PROP['worsensWith'],        RDFS.range,  PROP['Disease']))
g.add((PROP['recommendedFor'],     RDFS.domain, PROP['Food']))
g.add((PROP['recommendedFor'],     RDFS.range,  PROP['Disease']))
g.add((PROP['contraindicatedFor'], RDFS.domain, PROP['Food']))
g.add((PROP['contraindicatedFor'], RDFS.range,  PROP['Disease']))
g.add((PROP['predictedFor'],       RDFS.domain, PROP['Food']))
g.add((PROP['predictedFor'],       RDFS.range,  PROP['Disease']))


<Graph identifier=Nfd2d5274a308489bac2578a6e3883eee (<class 'rdflib.graph.Graph'>)>

In [12]:
# Verify schema is populated: list declared properties and their domain/range
results = g.query("""
    PREFIX prop: <http://nutrition-kg.org/property/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?prop ?domain ?range WHERE {
        ?prop rdfs:domain ?domain .
        ?prop rdfs:range  ?range  .
    }
""")
print("Declared properties (schema):")
for row in results:
    p = str(row.prop).split('/')[-1]
    d = str(row.domain).split('/')[-1]
    r = str(row.range).split('/')[-1]
    print(f"  prop:{p:22s}  domain:{d}  range:{r}")
print()
# Note: recommendedFor and contraindicatedFor are declared here as schema,
# but their data triples will be populated after the reasoning step (Section 6).

Declared properties (schema):
  prop:highIn                  domain:Food  range:Nutrient
  prop:recommendedFor          domain:Food  range:Disease
  prop:contraindicatedFor      domain:Food  range:Disease
  prop:predictedFor            domain:Food  range:Disease
  prop:treats                  domain:Nutrient  range:Disease
  prop:worsensWith             domain:Nutrient  range:Disease



## 5. Enrichment with Disease Data

Three external data sources are integrated to populate nutrient-disease relationships:

| Relation | Source | Evidence type |
|---|---|---|
| `highIn` | USDA FoodData Central | Per-food nutrient content, top-quartile threshold |
| `treats` | CTD (Comparative Toxicogenomics Database) | `DirectEvidence == 'therapeutic'` |
| `worsensWith` | Curated NIH/WHO/AHA/ADA dietary guidelines (`worsensWith_curated.csv`) | Clinical dietary recommendations |

**CTD therapeutic evidence** maps nutrients to diseases they clinically treat:
`(Iron) -- treats --> (Anemia)`

**Curated `worsensWith` pairs** encode nutrients whose excess dietary intake worsens specific
diseases, drawn from NIH ODS, WHO, AHA, ADA, and EASL clinical guidelines. The CTD
`marker/mechanism` evidence class was evaluated but discarded because it conflates diagnostic
biomarker associations (a nutrient measured as a disease marker in blood tests) with dietary
harm (excess intake causing or aggravating disease), making it unsuitable as an avoidance
signal.


In [13]:
ctd = pd.read_csv(
    CTD_CSV,
    comment='#',
    names=['ChemicalName', 'ChemicalID', 'CasRN',
           'DiseaseName', 'DiseaseID', 'DirectEvidence', 'PubMedIDs']
)
print(f"CTD total records: {len(ctd)}")

CTD total records: 109064


In [14]:
# Keep only therapeutic relationships
ctd_therapeutic = ctd[ctd['DirectEvidence'] == 'therapeutic'].copy()
print(f"Therapeutic associations: {len(ctd_therapeutic)}")

# Check which of our nutrients appear in CTD
ctd_therapeutic['ChemicalName_lower'] = ctd_therapeutic['ChemicalName'].str.lower()


Therapeutic associations: 39568


In [15]:
# Updated nutrient name mapping between our names and CTD names
nutrient_name_map = {
    'Zinc': 'Zinc',
    'Vitamin_D': 'Vitamin D',
    'Calcium': 'Calcium',
    'Magnesium': 'Magnesium',
    'Potassium': 'Potassium',
    'Sodium': 'Sodium',
    'Iron': 'Iron',
    'Fiber': 'Dietary Fiber',
}

# Get all matches with correct names
all_matches = []
for our_name, ctd_name in nutrient_name_map.items():
    results = ctd_therapeutic[
        ctd_therapeutic['ChemicalName'] == ctd_name
    ].copy()
    results['our_nutrient_name'] = our_name
    all_matches.append(results)

matches_final = pd.concat(all_matches)
print(f"Total therapeutic associations: {len(matches_final)}")
print(matches_final.groupby('our_nutrient_name')['DiseaseName'].count().sort_values(ascending=False))

Total therapeutic associations: 200
our_nutrient_name
Zinc         60
Vitamin_D    40
Calcium      31
Magnesium    30
Potassium    21
Sodium       11
Iron          6
Fiber         1
Name: DiseaseName, dtype: int64


In [16]:

DISEASE = Namespace("http://nutrition-kg.org/disease/")
g.bind("disease", DISEASE)

# Add disease triples to KG
for _, row in matches_final.iterrows():
    nutrient_uri = NUTRIENT[row['our_nutrient_name']]
    disease_name_clean = re.sub(r'[^a-zA-Z0-9_]', '_', row['DiseaseName'])
    disease_uri = DISEASE[disease_name_clean]
    
    # nutrient --treats--> disease
    g.add((nutrient_uri, PROP['treats'], disease_uri))
    
    # disease type triple
    g.add((disease_uri, RDF.type, PROP['Disease']))

print(f"Total triples now: {len(g)}")
print(f"\nSample disease triples:")
count = 0
for s, p, o in g.triples((None, PROP['treats'], None)):
    print(f"  {s.split('/')[-1]} -- treats --> {o.split('/')[-1]}")
    count += 1
    if count >= 10:
        break

Total triples now: 1758

Sample disease triples:
  Zinc -- treats --> Acrodermatitis
  Zinc -- treats --> Acute_Kidney_Injury
  Vitamin_D -- treats --> Acute_Kidney_Injury
  Calcium -- treats --> Acute_Kidney_Injury
  Magnesium -- treats --> Acute_Kidney_Injury
  Potassium -- treats --> Acute_Kidney_Injury
  Sodium -- treats --> Acute_Kidney_Injury
  Zinc -- treats --> Asthma
  Zinc -- treats --> Atherosclerosis
  Zinc -- treats --> Brain_Injuries


In [17]:
# Load curated worsensWith associations from NIH/WHO/AHA/ADA dietary guidelines
# Replaces CTD marker/mechanism which conflated diagnostic biomarkers with dietary harm
worsens_df = pd.read_csv(BASE_DIR / 'worsensWith_curated.csv')

print(f"Curated worsensWith pairs loaded: {len(worsens_df)}")
print(worsens_df.groupby('nutrient')['disease'].count().sort_values(ascending=False))

# Wipe any previously added worsensWith triples
for s, p, o in list(g.triples((None, PROP['worsensWith'], None))):
    g.remove((s, p, o))

# Add worsensWith triples to graph
worsens_added = 0
for _, row in worsens_df.iterrows():
    nutrient_uri = NUTRIENT[row['nutrient']]
    if (nutrient_uri, RDF.type, PROP['Nutrient']) not in g:
        continue
    disease_name_clean = re.sub(r'[^a-zA-Z0-9_]', '_', row['disease'])
    disease_uri = DISEASE[disease_name_clean]
    if (nutrient_uri, PROP['worsensWith'], disease_uri) not in g:
        g.add((nutrient_uri, PROP['worsensWith'], disease_uri))
        g.add((disease_uri, RDF.type, PROP['Disease']))
        worsens_added += 1

print(f"Added {worsens_added} worsensWith triples")
print(f"Total triples now: {len(g)}")

Curated worsensWith pairs loaded: 126
nutrient
Sodium           26
Saturated_Fat    17
Sugar            16
Cholesterol      10
Iron             10
Calcium           8
Fiber             8
Potassium         8
Protein           8
Vitamin_D         6
Magnesium         5
Zinc              4
Name: disease, dtype: int64
Added 126 worsensWith triples
Total triples now: 1919


In [18]:
# Summary of the KG
foods = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Food']))))
nutrients = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Nutrient']))))
diseases = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Disease']))))
food_nutrient_triples = len(list(g.triples((None, PROP['highIn'], None))))
treats_triples = len(list(g.triples((None, PROP['treats'], None))))
worsens_triples = len(list(g.triples((None, PROP['worsensWith'], None))))

print("=== Knowledge Graph Summary ===")
print(f"Foods:                       {foods}")
print(f"Nutrients:                   {nutrients}")
print(f"Diseases:                    {diseases}")
print(f"Food-nutrient triples:       {food_nutrient_triples}")
print(f"Nutrient-disease (treats):   {treats_triples}")
print(f"Nutrient-disease (worsens):  {worsens_triples}")
print(f"Total triples:               {len(g)}")


=== Knowledge Graph Summary ===
Foods:                       314
Nutrients:                   18
Diseases:                    188
Food-nutrient triples:       1055
Nutrient-disease (treats):   200
Nutrient-disease (worsens):  126
Total triples:               1919


In [19]:
# Save the KG to disk
g.serialize(destination=out_dir / 'nutrition_kg.ttl', format='turtle')
print("KG saved to nutrition_kg.ttl")

KG saved to nutrition_kg.ttl


## 6. Logical Reasoning

Three inference rules derive new food-disease relationships from the existing KG structure.

**Rule 1 — Recommendation:**
```
recommendedFor(F, D) :- highIn(F, N), treats(N, D).
```
If a food is *high in* nutrient N, and N *therapeutically treats* disease D → food is *recommendedFor* D.

**Rule 2 — Contraindication:**
```
contraindicatedFor(F, D) :- highIn(F, N), worsensWith(N, D).
```
If a food is *high in* nutrient N, and N *worsens* disease D → food is *contraindicatedFor* D.

**Rule 3 — Contradiction resolution:**
```
¬recommendedFor(F, D) :- contraindicatedFor(F, D).
```
If a food is both *recommendedFor* and *contraindicatedFor* the same disease, the *recommendedFor* triple is retracted (contraindication takes priority).

All three rules are fully implemented below.

### Implementation: Datalog Rules via SPARQL CONSTRUCT

The rules above are expressed in **Datalog** notation for clarity, but executed as
**SPARQL CONSTRUCT** queries in rdflib (forward-chaining materialization).

**Key distinctions between Datalog and SPARQL CONSTRUCT:**

| Feature | Full Datalog | SPARQL CONSTRUCT (used here) |
|---|---|---|
| Recursion | Yes (fixpoint semantics) | No |
| Existential quantification in head | Yes | No |
| Negation-as-failure | Yes (stratified) | Limited |
| Non-recursive conjunction | Yes | Yes ✓ |

For the *non-recursive, conjunction-only* rules used here, a single-pass SPARQL CONSTRUCT
is semantically equivalent to Datalog evaluation — both produce the same inferred triples.
A full Datalog engine (e.g., Soufflé, owlrl with RDFS/OWL closure) would be required if
recursive rules or OWL entailment reasoning were needed.

The Datalog notation is used as the **formal specification language**; SPARQL CONSTRUCT is
the **execution mechanism**.

In [20]:
# ── Rule 1: Recommendation ───────────────────────────────────────────────────
# Datalog form:  recommendedFor(F, D) :- highIn(F, N), treats(N, D).
recommend_rule = prepareQuery("""
    CONSTRUCT { ?food prop:recommendedFor ?disease }
    WHERE {
        ?food     prop:highIn ?nutrient .
        ?nutrient prop:treats ?disease  .
    }
""", initNs={"prop": PROP})

inferred_triples = []
for food_uri, _, disease_uri in g.query(recommend_rule):
    if (food_uri, PROP['recommendedFor'], disease_uri) not in g:
        g.add((food_uri, PROP['recommendedFor'], disease_uri))
        inferred_triples.append((
            str(food_uri).split('/')[-1],
            'recommendedFor',
            str(disease_uri).split('/')[-1]
        ))

print(f"Inferred {len(inferred_triples)} recommendedFor triples")

Inferred 13229 recommendedFor triples


In [21]:
# ── Rule 2: Contraindication ─────────────────────────────────────────────────
# Datalog form:  contraindicatedFor(F, D) :- highIn(F, N), worsensWith(N, D).
# worsensWith triples come from the curated NIH/WHO/AHA/ADA guidelines (worsensWith_curated.csv).
contraindication_rule = prepareQuery("""
    CONSTRUCT { ?food prop:contraindicatedFor ?disease }
    WHERE {
        ?food     prop:highIn      ?nutrient .
        ?nutrient prop:worsensWith ?disease  .
    }
""", initNs={"prop": PROP})


contra_count = 0
for food_uri, _, disease_uri in g.query(contraindication_rule):
    if (food_uri, PROP['contraindicatedFor'], disease_uri) not in g:
        g.add((food_uri, PROP['contraindicatedFor'], disease_uri))
        inferred_triples.append((
            str(food_uri).split('/')[-1],
            'contraindicatedFor',
            str(disease_uri).split('/')[-1]
        ))
        contra_count += 1

print(f"Inferred {contra_count} contraindicatedFor triples")


Inferred 6428 contraindicatedFor triples


In [22]:
# Find and remove ALL remaining contradictions properly
resolved = 0
contradictions_found = []

for food_uri, _, disease_uri in list(g.triples((None, PROP['recommendedFor'], None))):
    if (food_uri, PROP['contraindicatedFor'], disease_uri) in g:
        contradictions_found.append((food_uri, disease_uri))

print(f"Contradictions found: {len(contradictions_found)}")

# Remove recommendedFor where contraindication exists
for food_uri, disease_uri in contradictions_found:
    g.remove((food_uri, PROP['recommendedFor'], disease_uri))
    resolved += 1

print(f"Resolved: {resolved}")

# Verify
remaining = [
    (s, o) for s, _, o in g.triples((None, PROP['recommendedFor'], None))
    if (s, PROP['contraindicatedFor'], o) in g
]
print(f"Remaining contradictions: {len(remaining)}")


Contradictions found: 1463
Resolved: 1463


Remaining contradictions: 0


In [23]:
# Save updated KG
g.serialize(destination=out_dir / 'nutrition_kg_reasoned.ttl', format='turtle')
print("Reasoned KG saved!\n")

# Final summary
foods_count = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Food']))))
nutrients_count = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Nutrient']))))
diseases_count = len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Disease']))))
recommended = len(list(g.triples((None, PROP['recommendedFor'], None))))
contraindicated = len(list(g.triples((None, PROP['contraindicatedFor'], None))))

print("=== Final KG Summary ===")
print(f"Foods:                     {foods_count}")
print(f"Nutrients:                 {nutrients_count}")
print(f"Diseases:                  {diseases_count}")
print(f"recommendedFor triples:    {recommended}")
print(f"contraindicatedFor triples:{contraindicated}")
print(f"Total triples:             {len(g)}")

Reasoned KG saved!

=== Final KG Summary ===
Foods:                     314
Nutrients:                 18
Diseases:                  188
recommendedFor triples:    11766
contraindicatedFor triples:6428
Total triples:             20113


## 7. KG Embeddings — RotatE

**Why RotatE?**
RotatE (Sun et al., 2019) represents relations as rotations in complex vector space.
This gives it the ability to model *relation composition* — precisely the pattern used
in our KG: `highIn ∘ treats → recommendedFor`. TransE captures translation but not
rotation-based composition; ComplEx handles antisymmetry but less naturally encodes
composition chains. RotatE is therefore the most principled choice for this KG structure.

**Training / evaluation split design:**
- All base triples (`highIn`, `treats`, `worsensWith`) are split 80/10/10 into train/validation/test
- RotatE is evaluated on the same relations it trains on, giving meaningful MRR and Hits@10 metrics
- Inferred triples (`recommendedFor`, `contraindicatedFor`) are included in `tf_full` vocabulary only, so `predict_target()` can score them — they are not part of the train/validation/test split

**Honest evaluation notes (LO12 — KGs, ML, AI):**
- This is a **proof-of-concept** evaluation on a small KG (~550 entities, ~4k base triples).
  Metrics should not be compared to SOTA benchmarks on FB15k-237 or WN18RR.
- A rigorous study would include model comparison (TransE, ComplEx) and cross-validated
  hyperparameter search, which is beyond the scope of this course project.


In [24]:
# ── Triple preparation ────────────────────────────────────────────────────────
# ONE TriplesFactory built from ALL five relations the KG uses:
#   highIn, treats, worsensWith, recommendedFor, contraindicatedFor.
# It is built from the reasoned graph `g` (Section 6 has materialised the inferred
# recommendedFor / contraindicatedFor triples). We exclude rdf:type, all RDFS
# schema/label triples, and predictedFor.
#
# Why this matters: previously the two food-disease relations the advisor actually
# queries (recommendedFor, contraindicatedFor) were placed in the vocabulary only and
# never split into training. Their relation embeddings therefore never received a
# gradient and stayed at random initialisation, so predict_target() for those relations
# ranked foods through a random rotation. Training on all five relations fixes this.

kge_predicates = ['highIn', 'treats', 'worsensWith', 'recommendedFor', 'contraindicatedFor']

kge_triples_list = [
    (str(s).split('/')[-1], str(p).split('/')[-1], str(o).split('/')[-1])
    for s, p, o in g
    if str(p).split('/')[-1] in kge_predicates
]

kge_array = np.array(kge_triples_list)

# Single factory over all five relations - shared entity/relation vocabulary.
tf_full = TriplesFactory.from_labeled_triples(kge_array)

print(f"Total KGE triples (all 5 relations): {len(kge_array)}")
print("Predicate distribution (full graph):")
for pred, count in Counter(kge_array[:, 1]).most_common():
    print(f"  {pred}: {count}")

# 80/10/10 split across ALL five relations (so every relation appears in train).
train, valid, test = tf_full.split(
    ratios=[0.8, 0.1, 0.1],
    random_state=42,
)
tf = tf_full  # used by predict_target later

print(f"\nTraining triples:   {train.num_triples}")
print(f"Validation triples: {valid.num_triples}")
print(f"Test triples:       {test.num_triples}")

# Per-relation counts in the TRAIN split (sanity: all five must be > 0).
train_labeled = train.label_triples(train.mapped_triples)
print("\nPredicate distribution in TRAIN split:")
for pred, count in Counter(train_labeled[:, 1]).most_common():
    print(f"  {pred}: {count}")


Total KGE triples (all 5 relations): 19575
Predicate distribution (full graph):
  recommendedFor: 11766
  contraindicatedFor: 6428
  highIn: 1055
  treats: 200
  worsensWith: 126

Training triples:   15660
Validation triples: 1957
Test triples:       1958

Predicate distribution in TRAIN split:
  recommendedFor: 9384
  contraindicatedFor: 5185
  highIn: 837
  treats: 157
  worsensWith: 97


In [25]:
result = pipeline(
    model='RotatE',
    training=train,
    validation=valid,
    testing=test,
    model_kwargs=dict(embedding_dim=64),
    #since it runs on CPU, batch_size = 32 
    training_kwargs=dict(num_epochs=100, batch_size=256),
    random_seed=42,
)

print("Training done!")


No cuda devices were available. The model runs on CPU


C:\Users\serka\kg_nutrition\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cpu:   0%|          | 0.00/62.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/62.0 [00:00<?, ?batch/s]

...

Training batches on cpu:   0%|          | 0.00/62.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/62.0 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/1.96k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 1.61s seconds


Training done!


In [26]:
model = result.model
torch.save(model.state_dict(), out_dir / 'rotate_model.pt')
print("RotatE model saved.")

RotatE model saved.


In [27]:
# ── Per-relation evaluation ───────────────────────────────────────────────────
# After moving recommendedFor / contraindicatedFor into training, those two
# relations dominate the test set, so a single aggregate MRR would mask the
# base-relation (highIn / treats / worsensWith) performance. We therefore report
# PyKEEN's *filtered* ranking metrics PER RELATION.
from pykeen.evaluation import RankBasedEvaluator

num_entities = tf_full.num_entities
random_hits10 = 10 / num_entities
print(f"Entities: {num_entities}  |  random Hits@10 baseline ~= {random_hits10:.4f}\n")

all_filter = [train.mapped_triples, valid.mapped_triples, test.mapped_triples]
test_labeled = test.label_triples(test.mapped_triples)

per_rel_rows = []
for rel in kge_predicates:
    mask = test_labeled[:, 1] == rel
    n = int(mask.sum())
    if n == 0:
        per_rel_rows.append({'relation': rel, '#test': 0,
                             'Hits@1': float('nan'), 'Hits@10': float('nan'),
                             'MRR': float('nan')})
        continue
    rel_triples = test.mapped_triples[torch.as_tensor(mask)]
    evaluator = RankBasedEvaluator(filtered=True)
    res = evaluator.evaluate(
        model=model,
        mapped_triples=rel_triples,
        additional_filter_triples=all_filter,
        use_tqdm=False,
    )
    per_rel_rows.append({
        'relation': rel,
        '#test': n,
        'Hits@1': res.get_metric('hits_at_1'),
        'Hits@10': res.get_metric('hits_at_10'),
        'MRR': res.get_metric('inverse_harmonic_mean_rank'),
    })

per_rel_df = pd.DataFrame(per_rel_rows)
pd.set_option('display.float_format', lambda v: f'{v:.4f}')
print("Per-relation filtered ranking metrics (both sides, realistic):")
print(per_rel_df.to_string(index=False))

# Aggregate (all test triples) for reference.
agg_mrr = float(result.metric_results.get_metric('inverse_harmonic_mean_rank'))
agg_h10 = float(result.metric_results.get_metric('hits_at_10'))
print(f"\nAggregate over all {test.num_triples} test triples: "
      f"MRR={agg_mrr:.4f}, Hits@10={agg_h10:.4f}")
print("(Aggregate is dominated by recommendedFor/contraindicatedFor; "
      "see the per-relation table for the base relations.)")


Entities: 520  |  random Hits@10 baseline ~= 0.0192



INFO:pykeen.evaluation.evaluator:Evaluation took 0.35s seconds


INFO:pykeen.evaluation.evaluator:Evaluation took 0.29s seconds


INFO:pykeen.evaluation.evaluator:Evaluation took 0.30s seconds


INFO:pykeen.evaluation.evaluator:Evaluation took 1.14s seconds


INFO:pykeen.evaluation.evaluator:Evaluation took 0.67s seconds


Per-relation filtered ranking metrics (both sides, realistic):
          relation  #test  Hits@1  Hits@10    MRR
            highIn    110  0.7273   0.8773 0.7905
            treats     23  0.7826   0.9130 0.8163
       worsensWith     14  1.0000   1.0000 1.0000
    recommendedFor   1200  0.9654   0.9946 0.9764
contraindicatedFor    611  0.9092   0.9853 0.9388

Aggregate over all 1958 test triples: MRR=0.9525, Hits@10=0.9842
(Aggregate is dominated by recommendedFor/contraindicatedFor; see the per-relation table for the base relations.)


In [28]:
# Build lookup from clean URI back to original food name
uri_to_original = {}
for _, row in foundation_foods.iterrows():
    clean = clean_uri(row['description'])
    uri_to_original[clean] = row['description']

print(f"Lookup table built: {len(uri_to_original)} foods")

Lookup table built: 365 foods


## 8. Results: Dietary Advisor

This section combines logical reasoning and KG embeddings into a unified recommendation
service, demonstrating the neuro-symbolic integration (LO12) and KG service layer (LO11).

### How it works

`dietary_advice('Disease_Name')` returns:
1. **Recommended foods** — inferred by Rule 1: foods high in nutrients that treat the disease, ranked by a normalized nutrient-score
2. **Foods to avoid** — inferred by Rule 2: foods high in nutrients that worsen the disease
3. **Embedding-completed predictions** — the `predictedFor` edges from §8.4: foods unlocked when RotatE-predicted `highIn` edges are propagated through Rule 1, each shown with its nutrient bridge and labelled *unverified*

Both inferred relations (`recommendedFor` and `contraindicatedFor`) are now part of the RotatE
training split (Section 7), so both relation embeddings are genuinely learned. The earlier
asymmetric framing — that only `recommendedFor` carried "geometric signal" while
`contraindicatedFor` did not — was an artefact of `contraindicatedFor` never being trained, and
no longer applies: both relations were equally untrained before, and are equally trained now.

The advisor's predicted section now reads the **real `predictedFor` triples** materialised in
§8.4 (it queries the graph only — no `predict_target`/model call at advise time). These are
recommendations the symbolic pipeline could not derive: foods unlocked when RotatE-predicted
`highIn` edges are propagated through Rule 1, surfaced with their nutrient bridge and clearly
labelled unverified. Because §8.4 builds `predictedFor` disjoint from rule `recommendedFor` and
drops contraindicated pairs, the fencing is **structural, not coincidental** — a predicted food
can never appear in the recommend-by-rules or avoid lists. Avoidance is still *served* by the
logical rules, which are exact and complete for the single-hop contraindication pattern; the
per-relation evaluation (Section 7) reports how well the embedding approximates each relation.


Section 8.4 then introduces a separate, principled use of the embedding: because `recommendedFor` is a saturated materialization of Rule 1 (nothing to find), we instead complete the genuinely-lossy, threshold-defined `highIn` relation and propagate the recovered edges through Rule 1 to unlock new recommendations (`predictedFor`).

### Available diseases (sample)
`Anemia`, `Diabetes_Mellitus`, `Asthma`, `Atherosclerosis`, `Hypertension`,
`Osteoporosis`, `Obesity`, `Magnesium_Deficiency`, `Fever`

Use `print(all_diseases)` for the complete list of 150+ diseases.


In [29]:
# List all available diseases correctly
all_diseases = sorted([
    str(o).split('/')[-1].replace('_', ' ')
    for s, p, o in g.triples((None, PROP['treats'], None))
] + [
    str(o).split('/')[-1].replace('_', ' ')
    for s, p, o in g.triples((None, PROP['worsensWith'], None))
])

all_diseases = sorted(set(all_diseases))
print(f"Total diseases in KG: {len(all_diseases)}")
print("\nAll available diseases:")
for d in all_diseases:
    print(f"  {d}")

Total diseases in KG: 188

All available diseases:
  Acrodermatitis
  Acute Coronary Syndrome
  Acute Kidney Injury
  Alzheimer Disease
  Anemia
  Anemia  Hypochromic
  Anemia  Iron Deficiency
  Arrhythmias  Cardiac
  Arteriosclerosis
  Arthralgia
  Asthma
  Atherosclerosis
  Atrial Fibrillation
  Autistic Disorder
  Basal Cell Nevus Syndrome
  Bone Diseases
  Bone Diseases  Endocrine
  Bradycardia
  Brain Injuries
  COVID 19
  Carcinoma  Ehrlich Tumor
  Carcinoma  Squamous Cell
  Cardiomyopathies
  Cardiotoxicity
  Cardiovascular Diseases
  Cell Transformation  Neoplastic
  Cerebrovascular Disorders
  Chemical and Drug Induced Liver Injury
  Cholestasis
  Cholestasis  Extrahepatic
  Cognition Disorders
  Colitis
  Colonic Neoplasms
  Colorectal Neoplasms
  Constipation
  Copper Metabolism Disorders
  Coronary Disease
  Coronary Vasospasm
  Craniofacial Abnormalities
  Crohn Disease
  Developmental Disabilities
  Diabetes Mellitus
  Diabetes Mellitus  Type 1
  Diabetes Mellitus  Type 2

In [30]:
# Daily reference values for normalization (based on FDA daily values)
daily_reference = {
    'Sodium':        2300,
    'Potassium':     4700,
    'Calcium':       1300,
    'Iron':            18,
    'Magnesium':      420,
    'Zinc':            11,
    'Vitamin_D':       20,
    'Fiber':           28,
    'Protein':         50,
    'Fat':             78,
    'Carbohydrate':   275,
    'Sugar':           50,
    'Saturated_Fat':   20,
    'Vitamin_C':       90,
    'Vitamin_B6':       1.7,
    'Vitamin_B12':      2.4,
    'Folate':         400,
    'Cholesterol':    300,
}

def score_food_for_disease(food_name, disease_name):
    disease_uri = DISEASE[disease_name.replace(' ', '_')]
    score = 0
    
    food_row = df[df['food'] == food_name]
    
    for _, row in food_row.iterrows():
        nutrient = row['nutrient']
        amount = row['amount']
        nutrient_uri = NUTRIENT[nutrient]
        ref = daily_reference.get(nutrient, 100)
        normalized = amount / ref
        
        if (nutrient_uri, PROP['treats'], disease_uri) in g:
            score += normalized
        if (nutrient_uri, PROP['worsensWith'], disease_uri) in g:
            score -= normalized
    
    return round(score, 4)

In [31]:
def dietary_advice(disease_name):
    """
    Print dietary advice for a given disease.

    Parameters
    ----------
    disease_name : str
        Disease name using underscores for spaces, e.g. 'Diabetes_Mellitus',
        'Anemia', 'Hypertension'. Call print(all_diseases) for the full list.

    Output
    ------
    - Foods recommended by logical rules (high in a therapeutic nutrient)
    - Foods to avoid (high in a worsening nutrient)
    - Foods predicted via embedding-completed `highIn` (the §8.4 `predictedFor` edges):
      recommendations unlocked when RotatE-predicted `highIn` edges are propagated through
      Rule 1, shown with their nutrient bridge and labelled unverified. This section reads the
      real `predictedFor` triples from the graph `g` only — it does not call the model / tf at
      advise time. Because §8.4 builds `predictedFor` disjoint from rule `recommendedFor` and
      drops contraindicated pairs, a predicted food can never collide with the other two lists.
    """
    disease_uri = DISEASE[disease_name.replace(' ', '_')]

    # ── Logical recommendations (Rule 1) ─────────────────────────────────────
    recommended_raw = [
        uri_to_original.get(s.split('/')[-1], s.split('/')[-1].replace('_', ' '))
        for s, _, _ in g.triples((None, PROP['recommendedFor'], disease_uri))
    ]
    recommended = sorted(
        recommended_raw,
        key=lambda f: score_food_for_disease(f, disease_name),
        reverse=True
    )

    # ── Logical contraindications (Rule 2) ───────────────────────────────────
    contraindicated_raw = [
        uri_to_original.get(s.split('/')[-1], s.split('/')[-1].replace('_', ' '))
        for s, _, _ in g.triples((None, PROP['contraindicatedFor'], disease_uri))
    ]
    contraindicated = sorted(
        contraindicated_raw,
        key=lambda f: score_food_for_disease(f, disease_name),
        reverse=False  # worst foods first
    )

    # ── Embedding-completed predictions (the real predictedFor edges from §8.4) ──
    # For each predicted food, reconstruct the explainable bridge from the highInPredicted
    # provenance recorded in §8.4c: a nutrient N with both highInPredicted(food, N) and
    # treats(N, disease). Depends only on g.
    predicted = []
    for f_uri, _, _ in g.triples((None, PROP['predictedFor'], disease_uri)):
        food = str(f_uri).split('/')[-1]
        bridges = sorted(
            str(n_uri).split('/')[-1]
            for _, _, n_uri in g.triples((f_uri, PROP['highInPredicted'], None))
            if (n_uri, PROP['treats'], disease_uri) in g
        )
        food_disp = uri_to_original.get(food, food.replace('_', ' '))
        predicted.append((food_disp, bridges))
    predicted.sort(key=lambda x: x[0])

    # ── Output ────────────────────────────────────────────────────────────────
    dis_disp = disease_name.replace('_', ' ')
    print(f"╔══════════════════════════════════════════")
    print(f"║ Dietary advice for: {dis_disp}")
    print(f"╠══════════════════════════════════════════")
    print(f"║ Recommended by rules ({len(recommended)} foods, top 5):")
    for f in recommended[:5]:
        print(f"║    + {f}")
    print(f"║")
    print(f"║ Avoid ({len(contraindicated)} foods, top 5):")
    for f in contraindicated[:5]:
        print(f"║    - {f}")
    print(f"║")
    print(f"║ Additionally predicted via embedding-completed highIn ({len(predicted)}, unverified):")
    if predicted:
        for food_disp, bridges in predicted[:5]:
            nutr = bridges[0] if bridges else '?'
            print(f"║    ? {food_disp} — via predicted highIn({food_disp}, {nutr}); {nutr} treats {dis_disp}")
        if len(predicted) > 5:
            print(f"║    … and {len(predicted) - 5} more")
    else:
        print(f"║    No embedding-completed predictions")
    print(f"╚══════════════════════════════════════════")

# The advisor's predicted section reads the predictedFor edges materialised in §8.4 (below),
# so it is demonstrated after that section — see the advisor verification cell.


### 8.4 Experiment: embedding-completed `highIn` → rule propagation

`recommendedFor` is a closed-world materialization of Rule 1, so it is *saturated* — there is
nothing missing for the embedding to find. `highIn`, by contrast, is created by a hard
75th-percentile threshold on a **continuous** nutrient amount (Limitation 1). A food at the 74th
percentile is dropped even though it is arguably "high", so **missing `highIn` edges are real and
well-posed**. This experiment targets that lost information.

**Mechanism.** RotatE never sees the nutrient amount — only the binary `highIn` edge. It therefore
cannot know a food is *near* the cutoff; it can only predict `highIn(F,N)` from the food's *other*
nutrient edges (collaborative filtering over the food×nutrient structure). So we **measure**
whether its predictions land among the genuine near-misses rather than assuming they do.

**Procedure.**
1. Reconstruct the full continuous amount table and, per nutrient, the 60th/75th percentiles.
2. Define bands over novel `(food, nutrient)` candidates (KG food, key nutrient, no `highIn` edge,
   known amount): **near-miss band** = 60th ≤ amount < 75th; **low** = amount < 60th.
3. Predict novel `highIn` edges per nutrient with `predict_target`.
4. Measure **lift** = (band fraction among RotatE's top-K novel predictions) / (band base rate).
5. Add the accepted predicted `highIn` edges to a *copy* of the graph and re-fire Rule 1 unchanged.
6. **Redefine `predictedFor`** = a `recommendedFor(F,D)` not derivable from observed `highIn` but
   derivable once the predicted `highIn` edges are added — i.e. genuinely embedding-unlocked. This
   replaces the previous forced top-`TOP_N` enrichment (which was padding).

**Honesty caveats (kept explicit, not hidden).**
- The 75th-percentile cutoff is arbitrary and the binary edge discards the amount that would most
  directly justify a rescue. This is **approximate completion validated against a softer threshold
  (the 60–75th band), not against ground truth.** The 60th-percentile boundary is itself arbitrary.
- RotatE predicts from nutrient-profile similarity, not from proximity to the cutoff; the band
  evaluation *tests* — but does not guarantee — that these coincide.
- Evaluation is restricted to foods already in the KG; foods with zero `highIn` edges have no
  embedding and are out of scope.
- The advisor's predicted section consumes the `predictedFor` edges defined here, so the
  user-facing service delivers this principled, rule-propagated signal directly (rather than an
  inline `recommendedFor` score on a saturated relation).


In [32]:
# ── 8.4a  Reconstruct continuous amounts and define near-miss bands ───────────
# Rebuild the FULL per-food, per-nutrient amount table for foundation foods x key nutrients —
# the table that exists BEFORE the level=='high' filter in Section 3 (which discarded amounts).
full_amounts = (
    food_nutrient_df[
        food_nutrient_df['fdc_id'].isin(foundation_foods['fdc_id']) &
        food_nutrient_df['nutrient_id'].isin(key_nutrients.keys())
    ][['fdc_id', 'nutrient_id', 'amount']]
    .merge(foundation_foods, on='fdc_id')
)
full_amounts['nutrient'] = full_amounts['nutrient_id'].map(key_nutrients)
full_amounts['food_uri'] = full_amounts['description'].map(clean_uri)

# Per-nutrient 60th and 75th percentiles (75th == the existing high_threshold).
pct = full_amounts.groupby('nutrient')['amount'].quantile([0.60, 0.75]).unstack()
pct.columns = ['p60', 'p75']

# Food entities that actually have an embedding (typed Food => >=1 observed highIn edge).
food_entities = set(
    str(s).split('/')[-1] for s, _, _ in g.triples((None, RDF.type, PROP['Food']))
) & set(tf.entity_to_id)

# Classify every (food, nutrient) pair: observed-high / near-miss band / low (disjoint).
def _classify(row):
    p60, p75 = pct.loc[row['nutrient'], 'p60'], pct.loc[row['nutrient'], 'p75']
    if row['amount'] >= p75:
        return 'observed_high'
    if row['amount'] >= p60:
        return 'band'
    return 'low'
full_amounts['zone'] = full_amounts.apply(_classify, axis=1)

# Novel candidate universe: KG foods, no existing highIn edge (amount < p75), known amount.
cand = full_amounts[
    full_amounts['food_uri'].isin(food_entities) &
    full_amounts['zone'].isin(['band', 'low'])
].copy()
amount_zone = {(r.food_uri, r.nutrient): (r.amount, r.zone) for r in cand.itertuples()}

band_by_nutrient = cand[cand['zone'] == 'band'].groupby('nutrient').size()
low_by_nutrient  = cand[cand['zone'] == 'low'].groupby('nutrient').size()
obs_by_nutrient  = full_amounts[full_amounts['zone'] == 'observed_high'].groupby('nutrient').size()

summary_bands = pd.DataFrame({
    'observed_high': obs_by_nutrient,
    'band(60-75)':   band_by_nutrient,
    'low(<60)':      low_by_nutrient,
}).fillna(0).astype(int)
print("Per-nutrient pair counts (candidate universe = band + low):")
print(summary_bands.to_string())

n_obs  = int(summary_bands['observed_high'].sum())
n_band = int(cand[cand['zone'] == 'band'].shape[0])
n_low  = int(cand[cand['zone'] == 'low'].shape[0])
print(f"\nTotals: observed_high={n_obs}, band={n_band}, low={n_low}")

# Verification A: bands non-empty and disjoint.
band_pairs = set(cand[cand['zone'] == 'band'][['food_uri', 'nutrient']].itertuples(index=False, name=None))
low_pairs  = set(cand[cand['zone'] == 'low'][['food_uri', 'nutrient']].itertuples(index=False, name=None))
A2_pass = n_band > 0 and n_low > 0 and band_pairs.isdisjoint(low_pairs)
print(f"[A] {'PASS' if A2_pass else 'FAIL'} - band ({n_band}) and low ({n_low}) non-empty and disjoint")


Per-nutrient pair counts (candidate universe = band + low):
               observed_high  band(60-75)  low(<60)
nutrient                                           
Calcium                   89           44       173
Carbohydrate              81           44       157
Cholesterol               22           12        53
Fat                       86           50       165
Fiber                     47           27        80
Folate                    31           11        58
Iron                      89           52       165
Magnesium                 89           51       166
Potassium                 90           44       172
Protein                   89           52       165
Saturated_Fat             27           13        61
Sodium                    84           44       161
Sugar                     35            5        69
Vitamin_B12               16           10        38
Vitamin_B6                50           25        95
Vitamin_C                 28           12        42
Vita

In [33]:
# ── 8.4b  Predict novel highIn edges and measure concentration in the band ────
# RotatE sees only the binary highIn edges (NOT the amounts), so it predicts highIn from a food's
# other nutrient edges. We rank its NOVEL, evaluable predictions per nutrient and measure whether
# the top ones fall in the near-miss band more than chance.
KEY_NUTRIENTS = [n for n in key_nutrients.values() if n in tf.entity_to_id]

pred_rows = []   # (nutrient, food_uri, score, zone)
for nutr in KEY_NUTRIENTS:
    try:
        pdf = predict_target(model=model, head=None, relation='highIn',
                             tail=nutr, triples_factory=tf).df
    except Exception:
        continue
    for r in pdf.itertuples():
        key = (r.head_label, nutr)
        if key in amount_zone:               # novel + evaluable candidate (food, no edge, known amount)
            _amt, zone = amount_zone[key]
            pred_rows.append((nutr, r.head_label, float(r.score), zone))

pred_df = pd.DataFrame(pred_rows, columns=['nutrient', 'food', 'score', 'zone'])
# Rank novel predictions per nutrient by score (higher = more confident).
pred_df['rank'] = pred_df.groupby('nutrient')['score'].rank(ascending=False, method='first')

# Model rate: per nutrient take top-b_n predictions (b_n = that nutrient's band size), pool them.
# K = number actually selected (>= each nutrient may have fewer evaluable candidates than b_n).
hits, K = 0, 0
for nutr, b_n in band_by_nutrient.items():
    topk = pred_df[(pred_df['nutrient'] == nutr) & (pred_df['rank'] <= int(b_n))]
    hits += int((topk['zone'] == 'band').sum())
    K += int(topk.shape[0])

base_rate  = n_band / (n_band + n_low)
model_rate = hits / K if K else float('nan')
lift = model_rate / base_rate if base_rate else float('nan')
print(f"K (top predictions selected, per-nutrient top-band-size) = {K}")
print(f"Base rate  (band / all novel candidates) = {base_rate:.3f}")
print(f"Model rate (band among top-K predictions) = {model_rate:.3f}")
print(f"Lift = model_rate / base_rate = {lift:.2f}")

# Verification B (soft): lift >= ~1.5 => RotatE recovers near-misses better than chance.
B2_pass = (lift >= 1.5)
verdict = "PASS (lift >= 1.5)" if B2_pass else "INCONCLUSIVE (lift < 1.5)"
print(f"[B] {verdict} - RotatE {'concentrates' if B2_pass else 'does NOT clearly concentrate'} "
      f"its top predictions in the near-miss band")


K (top predictions selected, per-nutrient top-band-size) = 555
Base rate  (band / all novel candidates) = 0.216
Model rate (band among top-K predictions) = 0.422
Lift = model_rate / base_rate = 1.95
[B] PASS (lift >= 1.5) - RotatE concentrates its top predictions in the near-miss band


In [34]:
# ── 8.4c  Propagate accepted highIn predictions through Rule 1 ────────────────
# Accept the top-ACCEPT_K novel highIn predictions per nutrient (embedding-driven, BLIND to the
# amount), add them to a COPY of the reasoned graph, and re-fire Rule 1 (unchanged) to derive
# recommendedFor triples the symbolic pipeline could not.
ACCEPT_K = 10

# Provenance for the integrity assertions (Verification D): capture ORIGINAL state of main graph.
orig_highin   = set(g.triples((None, PROP['highIn'], None)))
orig_rec      = set(g.triples((None, PROP['recommendedFor'], None)))

# Pure Rule-1 closure over the OBSERVED highIn edges (independent of the contradiction removal
# in Section 6 — Rule 1 re-derives the full set since highIn/treats are intact). This is the
# correct baseline to diff against, so new_rec isolates the effect of the PREDICTED edges only.
rec_observed = set(g.query(recommend_rule))

# Augmented copy: observed graph + accepted predicted highIn edges.
g_aug = Graph()
for t in g:
    g_aug.add(t)

accepted = []   # (food_uri, nutrient)
for nutr in KEY_NUTRIENTS:
    topk = pred_df[(pred_df['nutrient'] == nutr) & (pred_df['rank'] <= ACCEPT_K)]
    for r in topk.itertuples():
        accepted.append((r.food, nutr))
        g_aug.add((FOOD[r.food], PROP['highIn'], NUTRIENT[nutr]))

rec_aug = set(g_aug.query(recommend_rule))
new_rec = rec_aug - rec_observed              # embedding-unlocked recommendations

print(f"Accepted predicted highIn edges: {len(accepted)} (top-{ACCEPT_K}/nutrient)")
print(f"New recommendedFor triples unlocked by propagation: {len(new_rec)}")

# ── Redefine predictedFor on the MAIN KG ──────────────────────────────────────
# Clear any stale predictedFor / highInPredicted, then add the embedding-unlocked
# recommendations (skipping rule-contraindicated pairs so served lists stay consistent).
for t in list(g.triples((None, PROP['predictedFor'], None))):
    g.remove(t)
for t in list(g.triples((None, PROP['highInPredicted'], None))):
    g.remove(t)

# Lightweight schema for the new provenance property.
g.add((PROP['highInPredicted'], RDFS.label, Literal("embedding-predicted high-in-nutrient edge")))
g.add((PROP['highInPredicted'], RDFS.domain, PROP['Food']))
g.add((PROP['highInPredicted'], RDFS.range,  PROP['Nutrient']))

added_pf = 0
for f_uri, _, d_uri in new_rec:
    if (f_uri, PROP['contraindicatedFor'], d_uri) in g:
        continue
    g.add((f_uri, PROP['predictedFor'], d_uri))
    added_pf += 1
# Record predicted source edges for provenance / visualization.
for f, nutr in accepted:
    g.add((FOOD[f], PROP['highInPredicted'], NUTRIENT[nutr]))

foods_touched = set(str(s).split('/')[-1] for s, _, _ in g.triples((None, PROP['predictedFor'], None)))
dis_touched   = set(str(o).split('/')[-1] for _, _, o in g.triples((None, PROP['predictedFor'], None)))
print(f"predictedFor triples added to KG: {added_pf}")
print(f"Distinct foods gaining a recommendation:    {len(foods_touched)}")
print(f"Distinct diseases gaining a recommendation: {len(dis_touched)}")

# ── 5 worked examples (explainable form) ──────────────────────────────────────
treats_by_disease = {}
for n_uri, _, d_uri in g.triples((None, PROP['treats'], None)):
    treats_by_disease.setdefault(str(d_uri), set()).add(str(n_uri).split('/')[-1])
pred_high_by_food = {}
for f_uri, _, n_uri in g.triples((None, PROP['highInPredicted'], None)):
    pred_high_by_food.setdefault(str(f_uri), set()).add(str(n_uri).split('/')[-1])

print("\nWorked examples (embedding-unlocked recommendations):")
shown = 0
for f_uri, _, d_uri in sorted(new_rec, key=lambda x: (str(x[2]), str(x[0]))):
    if (f_uri, PROP['contraindicatedFor'], d_uri) in g:
        continue
    bridges = pred_high_by_food.get(str(f_uri), set()) & treats_by_disease.get(str(d_uri), set())
    if not bridges:
        continue
    nutr = sorted(bridges)[0]
    food = str(f_uri).split('/')[-1]
    food_disp = uri_to_original.get(food, food.replace('_', ' '))
    dis_disp = str(d_uri).split('/')[-1].replace('_', ' ')
    print(f"  - {food_disp}  recommendedFor  {dis_disp}")
    print(f"      via predicted highIn({food_disp}, {nutr}), and {nutr} treats {dis_disp}")
    shown += 1
    if shown >= 5:
        break

# ── Verification C and D ──────────────────────────────────────────────────────
C2_pass = added_pf > 0
print(f"\n[C] {'PASS' if C2_pass else 'FAIL'} - propagation produced {added_pf} new recommendation(s)"
      + ("" if C2_pass else " (likely: top-K predicted highIn hit no treats-bearing nutrient)"))

cur_highin = set(g.triples((None, PROP['highIn'], None)))
cur_rec    = set(g.triples((None, PROP['recommendedFor'], None)))
pf_pairs   = set((s, o) for s, _, o in g.triples((None, PROP['predictedFor'], None)))
rec_pairs  = set((s, o) for s, _, o in g.triples((None, PROP['recommendedFor'], None)))
D2_pass = (cur_highin == orig_highin) and (cur_rec == orig_rec) and pf_pairs.isdisjoint(rec_pairs)
print(f"[D] {'PASS' if D2_pass else 'FAIL'} - observed highIn ({len(cur_highin)}) and rule "
      f"recommendedFor ({len(cur_rec)}) unchanged; predictedFor disjoint from recommendedFor")

print("\n========= highIn-COMPLETION EXPERIMENT SUMMARY =========")
print(f"  A (bands valid)            : {'PASS' if A2_pass else 'FAIL'}  (band={n_band}, low={n_low})")
print(f"  B (band concentration)     : {'PASS' if B2_pass else 'INCONCLUSIVE'}  (lift={lift:.2f})")
print(f"  C (propagation unlocks new): {'PASS' if C2_pass else 'FAIL'}  ({added_pf} predictedFor)")
print(f"  D (observed data intact)   : {'PASS' if D2_pass else 'FAIL'}")
print("========================================================")


Accepted predicted highIn edges: 180 (top-10/nutrient)
New recommendedFor triples unlocked by propagation: 1506
predictedFor triples added to KG: 1386
Distinct foods gaining a recommendation:    55
Distinct diseases gaining a recommendation: 147

Worked examples (embedding-unlocked recommendations):
  - Cheese, pasteurized process, American, vitamin D fortified  recommendedFor  Acrodermatitis
      via predicted highIn(Cheese, pasteurized process, American, vitamin D fortified, Zinc), and Zinc treats Acrodermatitis
  - Chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised  recommendedFor  Acrodermatitis
      via predicted highIn(Chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised, Zinc), and Zinc treats Acrodermatitis
  - Chicken, drumstick, meat and skin, raw  recommendedFor  Acrodermatitis
      via predicted highIn(Chicken, drumstick, meat and skin, raw, Zinc), and Zinc treats Acrodermatitis
  - Corn flour, masa harina, wh

In [35]:
# ── Advisor verification: predicted section surfaces the real predictedFor edges ──
# For Acrodermatitis / Anemia / Hypertension confirm that dietary_advice()'s predicted
# section is exactly the §8.4 predictedFor triples, that each has a valid nutrient bridge,
# and that no predicted food leaks into the "recommended by rules" list.
import io, re
from contextlib import redirect_stdout

EXPECTED = {'Acrodermatitis': 10, 'Anemia': 3, 'Hypertension': 1}
all_pass = True

for disease, exp_n in EXPECTED.items():
    duri = DISEASE[disease]
    pf_foods   = {str(s).split('/')[-1] for s, _, _ in g.triples((None, PROP['predictedFor'], duri))}
    rule_foods = {str(s).split('/')[-1] for s, _, _ in g.triples((None, PROP['recommendedFor'], duri))}

    # Capture the advisor output to tie the printed heading to the graph.
    buf = io.StringIO()
    with redirect_stdout(buf):
        dietary_advice(disease)
    text = buf.getvalue()
    print(text)

    m = re.search(r"embedding-completed highIn \((\d+), unverified\)", text)
    heading_count = int(m.group(1)) if m else -1
    printed_q = text.count("║    ? ")

    # 1) advisor's predicted set == predictedFor triples, with expected count.
    eq = (len(pf_foods) == exp_n) and (heading_count == len(pf_foods))
    # 2) each predicted food has >=1 valid nutrient bridge (highInPredicted + treats).
    def _has_bridge(f):
        fu = FOOD[f]
        return any((n, PROP['treats'], duri) in g
                   for _, _, n in g.triples((fu, PROP['highInPredicted'], None)))
    bridged = all(_has_bridge(f) for f in pf_foods) and len(pf_foods) > 0
    # 3) no predicted food appears in the rules list.
    disjoint = pf_foods.isdisjoint(rule_foods)
    # printed lines are a subset of the predictedFor set (cap of 5 respected).
    printed_ok = printed_q == min(5, len(pf_foods))

    ok = eq and bridged and disjoint and printed_ok
    all_pass = all_pass and ok
    print(f"[{disease}] predictedFor={len(pf_foods)} (expected {exp_n}), heading={heading_count}, "
          f"printed={printed_q}")
    print(f"   count==expected & heading match : {'PASS' if eq else 'FAIL'}")
    print(f"   every predicted food has bridge : {'PASS' if bridged else 'FAIL'}")
    print(f"   disjoint from rule recommend    : {'PASS' if disjoint else 'FAIL'}")
    print(f"   printed <=5 lines, subset of set: {'PASS' if printed_ok else 'FAIL'}\n")

print("================ ADVISOR VERIFICATION ================")
print(f"  Overall: {'PASS' if all_pass else 'FAIL'}")
print("=====================================================")


╔══════════════════════════════════════════
║ Dietary advice for: Acrodermatitis
╠══════════════════════════════════════════
║ Recommended by rules (89 foods, top 5):
║    + Egg, yolk, dried
║    + Seeds, pumpkin seeds (pepitas), raw
║    + Sesame butter, creamy
║    + Seeds, sunflower seed kernels, dry roasted, with salt added
║    + Wild rice, dry, raw
║
║ Avoid (0 foods, top 5):
║
║ Additionally predicted via embedding-completed highIn (10, unverified):
║    ? Cheese, pasteurized process, American, vitamin D fortified — via predicted highIn(Cheese, pasteurized process, American, vitamin D fortified, Zinc); Zinc treats Acrodermatitis
║    ? Chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised — via predicted highIn(Chicken, broiler or fryers, breast, skinless, boneless, meat only, cooked, braised, Zinc); Zinc treats Acrodermatitis
║    ? Chicken, drumstick, meat and skin, raw — via predicted highIn(Chicken, drumstick, meat and skin, raw, Zinc); Zinc trea

╔══════════════════════════════════════════
║ Dietary advice for: Hypertension
╠══════════════════════════════════════════
║ Recommended by rules (122 foods, top 5):
║    + Flour, soy, defatted
║    + Chia seeds, dry, raw
║    + Seeds, pumpkin seeds (pepitas), raw
║    + Flaxseed, ground
║    + Flour, soy, full-fat
║
║ Avoid (84 foods, top 5):
║    - Salt, table, iodized
║    - Anchovies, canned in olive oil, with salt, drained
║    - Pork, cured, bacon, cooked, restaurant
║    - Olives, green, Manzanilla, stuffed with pimiento
║    - Ketchup, restaurant
║
║ Additionally predicted via embedding-completed highIn (1, unverified):
║    ? Beef, loin, tenderloin roast, separable lean only, boneless, trimmed to 0" fat, select, cooked, roasted — via predicted highIn(Beef, loin, tenderloin roast, separable lean only, boneless, trimmed to 0" fat, select, cooked, roasted, Magnesium); Magnesium treats Hypertension
╚══════════════════════════════════════════

[Hypertension] predictedFor=1 (expecte

In [36]:
# ── Serialize final KG ────────────────────────────────────────────────────────
g.serialize(destination=out_dir / 'nutrition_kg_final.ttl', format='turtle')

# ── Export triples to CSV ─────────────────────────────────────────────────────
triples_clean = [
    (str(s).split('/')[-1], str(p).split('/')[-1], str(o).split('/')[-1])
    for s, p, o in g
]
triples_df = pd.DataFrame(triples_clean, columns=['subject', 'predicate', 'object'])
triples_df.to_csv(out_dir / 'triples.csv', index=False)

# ── Write summary JSON ────────────────────────────────────────────────────────
summary = {
    'foods':                 len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Food'])))),
    'nutrients':             len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Nutrient'])))),
    'diseases':              len(set(s for s, p, o in g.triples((None, RDF.type, PROP['Disease'])))),
    'total_triples':         len(g),
    'predicted_for_triples': len(list(g.triples((None, PROP['predictedFor'], None)))),
    'mrr':                   float(result.metric_results.get_metric('inverse_harmonic_mean_rank')),
    'hits_at_10':            float(result.metric_results.get_metric('hits_at_10')),
}
with open(out_dir / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("All outputs saved to:", out_dir)
print(f"  nutrition_kg_final.ttl  ({len(g)} triples)")
print(f"  triples.csv             ({len(triples_df)} rows)")
print(f"  summary.json            MRR={summary['mrr']:.3f}, Hits@10={summary['hits_at_10']:.3f}")


All outputs saved to: C:\Users\serka\kg_nutrition\outputs
  nutrition_kg_final.ttl  (21682 triples)
  triples.csv             (21682 rows)
  summary.json            MRR=0.953, Hits@10=0.984


### Verification of the embedding fix

The following cells empirically verify that the two food–disease relations
(`recommendedFor`, `contraindicatedFor`) are now genuinely trained, rather than left at
random initialisation as in the earlier design. Each test prints **PASS** / **FAIL**.

- **A — Relations are in training** (necessary condition): both inferred relations have > 0
  triples in the train split.
- **B — The relation embeddings actually moved** (core test): for a sample of diseases we rank
  *all* foods by `predict_target(relation='recommendedFor', tail=disease)` and measure the
  overlap of the top-N with the rule-derived `recommendedFor` set. Since `recommendedFor` is a
  deterministic function of `highIn ∧ treats`, a correctly trained RotatE should recover it
  (high overlap), whereas the old random-rotation setup would score near a random/degree baseline.
- **C — Per-relation ranking beats random**: each relation's filtered Hits@10 clearly exceeds
  the random baseline (≈ 10 / num_entities).
- **D — Contradiction sanity** (behavioural): for several diseases we print the rule-recommend,
  rule-avoid and RotatE-predicted lists and confirm no food appears in both a recommend and an
  avoid list. We also revisit the old "hazelnuts in both lists for Hypertension" case now that
  `contraindicatedFor` is trained.


In [37]:
# ── Verification A: inferred relations are in the TRAIN split ─────────────────
train_rel_counts = Counter(train_labeled[:, 1])
print("Training triple counts per relation:")
for rel in kge_predicates:
    print(f"  {rel}: {train_rel_counts.get(rel, 0)}")

a_rec   = train_rel_counts.get('recommendedFor', 0)
a_contra = train_rel_counts.get('contraindicatedFor', 0)
A_pass = a_rec > 0 and a_contra > 0
print(f"\n[A] recommendedFor train triples={a_rec}, contraindicatedFor train triples={a_contra}")
print(f"[A] {'PASS' if A_pass else 'FAIL'} - both inferred relations appear in training")


Training triple counts per relation:
  highIn: 837
  treats: 157
  worsensWith: 97
  recommendedFor: 9384
  contraindicatedFor: 5185

[A] recommendedFor train triples=9384, contraindicatedFor train triples=5185
[A] PASS - both inferred relations appear in training


In [38]:
# ── Verification B: did the recommendedFor embedding actually move? ───────────
# For a sample of diseases, rank ALL foods by predict_target(recommendedFor, tail=disease)
# WITHOUT filtering known foods, and measure overlap@N with the rule-derived set.
# Compare against two interpretable baselines: random ordering and degree (most-connected
# foods first). A trained RotatE should recover the deterministic rule (overlap >> baseline);
# the old random-rotation setup would sit at ~the random baseline.
import numpy as _np

OVERLAP_N = 10
rng = _np.random.default_rng(42)

all_food_uris = [str(s).split('/')[-1] for s, p, o in g.triples((None, RDF.type, PROP['Food']))]
all_food_set = set(all_food_uris)

# Food degree (number of highIn edges) for the degree baseline.
food_degree = Counter(
    str(s).split('/')[-1] for s, p, o in g.triples((None, PROP['highIn'], None))
)
foods_by_degree = [f for f, _ in food_degree.most_common()]
foods_by_degree += [f for f in all_food_uris if f not in food_degree]  # zero-degree foods last

# Diseases that actually have a rule-derived recommendedFor set.
rec_by_disease = {}
for s, _, o in g.triples((None, PROP['recommendedFor'], None)):
    rec_by_disease.setdefault(str(o).split('/')[-1], set()).add(str(s).split('/')[-1])

eligible = [d for d, foods in rec_by_disease.items() if len(foods) >= 1]
sample_diseases = sorted(eligible)[:25]  # deterministic sample

model_overlaps, rand_overlaps, deg_overlaps = [], [], []
for d in sample_diseases:
    rule_set = rec_by_disease[d]
    # Model ranking
    try:
        pdf = predict_target(model=model, head=None, relation='recommendedFor',
                             tail=d, triples_factory=tf).df
        pdf = pdf[pdf['head_label'].isin(all_food_set)]
        top_model = list(pdf['head_label'].head(OVERLAP_N))
    except Exception:
        continue
    model_overlaps.append(len(set(top_model) & rule_set) / OVERLAP_N)
    # Random baseline (averaged over a few shuffles)
    r = []
    for _ in range(5):
        perm = list(all_food_uris)
        rng.shuffle(perm)
        r.append(len(set(perm[:OVERLAP_N]) & rule_set) / OVERLAP_N)
    rand_overlaps.append(float(_np.mean(r)))
    # Degree baseline
    deg_overlaps.append(len(set(foods_by_degree[:OVERLAP_N]) & rule_set) / OVERLAP_N)

mean_model = float(_np.mean(model_overlaps))
mean_rand  = float(_np.mean(rand_overlaps))
mean_deg   = float(_np.mean(deg_overlaps))

print(f"Sampled diseases with a rule set: {len(model_overlaps)}")
print(f"Mean overlap@{OVERLAP_N}  RotatE = {mean_model:.3f}")
print(f"Mean overlap@{OVERLAP_N}  random = {mean_rand:.3f}")
print(f"Mean overlap@{OVERLAP_N}  degree = {mean_deg:.3f}")

B_pass = mean_model >= 0.5 and mean_model > max(mean_rand, mean_deg) + 0.2
print(f"\n[B] {'PASS' if B_pass else 'FAIL'} - RotatE overlap@{OVERLAP_N} "
      f"({mean_model:.3f}) {'clearly exceeds' if B_pass else 'does NOT clearly exceed'} "
      f"baselines (random {mean_rand:.3f}, degree {mean_deg:.3f})")


Sampled diseases with a rule set: 25
Mean overlap@10  RotatE = 1.000
Mean overlap@10  random = 0.253
Mean overlap@10  degree = 0.744

[B] PASS - RotatE overlap@10 (1.000) clearly exceeds baselines (random 0.253, degree 0.744)


In [39]:
# ── Verification C: per-relation ranking beats random ─────────────────────────
# Uses per_rel_df from Section 7. Each relation's filtered Hits@10 must clearly exceed
# the random baseline (~10 / num_entities).
baseline = 10 / tf_full.num_entities
c_tbl = per_rel_df.copy()
c_tbl['random_Hits@10'] = baseline
c_tbl['PASS'] = c_tbl['Hits@10'] > 5 * baseline   # "clearly exceeds": >= 5x random
print(f"Random Hits@10 baseline ~= {baseline:.4f}  (PASS = Hits@10 > 5x baseline)\n")
print(c_tbl.to_string(index=False))

C_pass = bool(c_tbl.loc[c_tbl['#test'] > 0, 'PASS'].all())
rec_h10 = float(per_rel_df.loc[per_rel_df['relation'] == 'recommendedFor', 'Hits@10'].iloc[0])
print(f"\n[C] recommendedFor Hits@10 = {rec_h10:.3f}")
print(f"[C] {'PASS' if C_pass else 'FAIL'} - every relation with test triples beats the random baseline")


Random Hits@10 baseline ~= 0.0192  (PASS = Hits@10 > 5x baseline)

          relation  #test  Hits@1  Hits@10    MRR  random_Hits@10  PASS
            highIn    110  0.7273   0.8773 0.7905          0.0192  True
            treats     23  0.7826   0.9130 0.8163          0.0192  True
       worsensWith     14  1.0000   1.0000 1.0000          0.0192  True
    recommendedFor   1200  0.9654   0.9946 0.9764          0.0192  True
contraindicatedFor    611  0.9092   0.9853 0.9388          0.0192  True

[C] recommendedFor Hits@10 = 0.995
[C] PASS - every relation with test triples beats the random baseline


In [40]:
# ── Verification D: contradiction sanity (behavioural) ────────────────────────
# For several diseases print rule-recommend, rule-avoid and RotatE-recommend lists and
# confirm no food appears in both a recommend and an avoid list. We ALSO re-test the
# now-trained contraindicatedFor relation (the old "hazelnuts in both lists for
# Hypertension" failure), to check whether it is gone.
check_diseases = ['Hypertension', 'Diabetes_Mellitus', 'Osteoporosis', 'Anemia']

def _clean_set(pred, disease_uri):
    return set(s.split('/')[-1] for s, _, _ in g.triples((None, PROP[pred], disease_uri)))

def _rotate_top(rel, disease, k=5):
    try:
        pdf = predict_target(model=model, head=None, relation=rel,
                             tail=disease, triples_factory=tf).df
        pdf = pdf[pdf['head_label'].isin(all_food_set)]
        return list(pdf['head_label'].head(k))
    except Exception:
        return []

any_contradiction = False
for d in check_diseases:
    duri = DISEASE[d]
    rule_rec = _clean_set('recommendedFor', duri)
    rule_avd = _clean_set('contraindicatedFor', duri)
    rot_rec = _rotate_top('recommendedFor', d, 5)
    rot_avd = _rotate_top('contraindicatedFor', d, 5)  # re-test trained avoidance relation

    # Served recommendations = RotatE recs filtered to exclude rule-contraindicated foods.
    served_rec = [f for f in rot_rec if f not in rule_avd]

    print(f"\n=== {d.replace('_',' ')} ===")
    print(f"  rule recommend (n={len(rule_rec)}): {sorted(list(rule_rec))[:5]}")
    print(f"  rule avoid     (n={len(rule_avd)}): {sorted(list(rule_avd))[:5]}")
    print(f"  RotatE recommend top5: {rot_rec}")
    print(f"  RotatE avoid    top5 : {rot_avd}")
    print(f"  SERVED recommend (rotate, contra-filtered): {served_rec}")

    # Contradiction = a food in both a served recommend list and the avoid list.
    served_overlap = (set(rule_rec) | set(served_rec)) & rule_avd
    # Diagnostic: would the UNFILTERED trained avoidance relation collide with recommendations?
    rotate_self_overlap = set(rot_rec) & set(rot_avd)
    if served_overlap:
        any_contradiction = True
        print(f"  !! CONTRADICTION in served lists: {sorted(served_overlap)}")
    else:
        print(f"  OK: no food in both a served recommend and the avoid list")
    if rotate_self_overlap:
        print(f"  (diagnostic) RotatE recommend & RotatE avoid share: {sorted(rotate_self_overlap)}")

D_pass = not any_contradiction
print(f"\n[D] {'PASS' if D_pass else 'FAIL'} - served recommend and avoid lists never share a food")

# ── Overall summary ───────────────────────────────────────────────────────────
print("\n================ VERIFICATION SUMMARY ================")
print(f"  A (relations trained)            : {'PASS' if A_pass else 'FAIL'}")
print(f"  B (recommendedFor embedding moved): {'PASS' if B_pass else 'FAIL'}  "
      f"(overlap@{OVERLAP_N}={mean_model:.3f} vs baseline {max(mean_rand, mean_deg):.3f})")
print(f"  C (per-relation beats random)    : {'PASS' if C_pass else 'FAIL'}")
print(f"  D (no contradictions)            : {'PASS' if D_pass else 'FAIL'}")
print("======================================================")



=== Hypertension ===
  rule recommend (n=122): ['Almond_butter_creamy', 'Almond_milk_unsweetened_plain_refrigerated', 'Almond_milk_unsweetened_plain_shelf_stable', 'Arugula_baby_raw', 'Avocado_Hass_peeled_raw']
  rule avoid     (n=84): ['Anchovies_canned_in_olive_oil_with_salt_drained', 'Beans_black_canned_sodium_added_drained_and_rinsed', 'Beans_cannellini_canned_sodium_added_drained_and_rinsed', 'Beans_great_northern_canned_sodium_added_drained_and_rinsed', 'Beans_kidney_dark_red_canned_sodium_added_sugar_added_drained_and_rinsed']
  RotatE recommend top5: ['Nuts_hazelnuts_or_filberts_raw', 'Collards_raw', 'Flour_soy_fullfat', 'Nuts_pistachio_nuts_raw', 'Beans_cannellini_dry']
  RotatE avoid    top5 : ['Beets_raw', 'Tomato_juice_with_added_ingredients_from_concentrate_shelf_stable', 'Onion_rings_breaded_par_fried_frozen_prepared_heated_in_oven', 'Beans_pinto_canned_sodium_added_drained_and_rinsed', 'Eggs_Grade_A_Large_egg_whole']
  SERVED recommend (rotate, contra-filtered): ['Nuts_

### Optional experiment: generalization beyond the rule (held-out `treats` edges)

This is a **separate** experiment (it does not touch the main split, the reasoned graph `g`,
or the trained `model`). Before reasoning, we hold out a sample of `treats` edges so the
symbolic rule `recommendedFor(F,D) :- highIn(F,N), treats(N,D)` *physically cannot* derive the
corresponding `recommendedFor` triples. We re-derive the inferred relations from the reduced
edge set, train a fresh out-of-the-box RotatE on it (same hyperparameters), and then check
whether RotatE's `recommendedFor` predictions still surface the foods tied to the held-out
nutrient–disease pairs — i.e. food–disease links the rule could no longer produce. A positive
hit rate is direct evidence of the embedding generalising beyond the symbolic rule, which is
the project's stated goal. Set `RUN_HELDOUT = False` to skip (it retrains a second model).


In [41]:
# ── Optional held-out generalization experiment ──────────────────────────────
RUN_HELDOUT = True

if RUN_HELDOUT:
    import numpy as _np
    _rng = _np.random.default_rng(7)

    # 1. Collect BASE edges from the reasoned graph (we only read; g is untouched).
    highin = [(str(s).split('/')[-1], str(o).split('/')[-1])
              for s, _, o in g.triples((None, PROP['highIn'], None))]
    treats = [(str(s).split('/')[-1], str(o).split('/')[-1])
              for s, _, o in g.triples((None, PROP['treats'], None))]
    worsens = [(str(s).split('/')[-1], str(o).split('/')[-1])
               for s, _, o in g.triples((None, PROP['worsensWith'], None))]

    foods_high_in = {}          # nutrient -> set(foods)
    for f, n in highin:
        foods_high_in.setdefault(n, set()).add(f)

    # 2. Hold out ~15 treats edges (only ones that actually generate recommendations).
    candidate_treats = [t for t in treats if foods_high_in.get(t[0])]
    n_hold = min(15, len(candidate_treats))
    idx = _rng.choice(len(candidate_treats), size=n_hold, replace=False)
    held_treats = set(candidate_treats[i] for i in idx)
    kept_treats = [t for t in treats if t not in held_treats]
    print(f"Held out {len(held_treats)} of {len(treats)} treats edges.")

    # 3. Re-derive recommendedFor / contraindicatedFor from the REDUCED edge set.
    def derive_rec(treats_edges):
        rec = set()
        for n, d in treats_edges:
            for f in foods_high_in.get(n, ()):
                rec.add((f, d))
        return rec

    rec_full  = derive_rec(treats)        # what the full rule would produce
    rec_train = derive_rec(kept_treats)   # what the reduced rule produces
    contra = set()
    for n, d in worsens:
        for f in foods_high_in.get(n, ()):
            contra.add((f, d))
    # Resolve contradictions the same way Section 6 does (drop rec where contra exists).
    rec_train = {p for p in rec_train if p not in contra}
    rec_full  = {p for p in rec_full  if p not in contra}

    # Pairs the reduced rule can no longer derive (the held-out generalization targets).
    held_pairs = rec_full - rec_train
    held_by_disease = {}
    for f, d in held_pairs:
        held_by_disease.setdefault(d, set()).add(f)
    print(f"Held-out recommendedFor (food,disease) pairs the reduced rule cannot derive: "
          f"{len(held_pairs)} across {len(held_by_disease)} diseases.")

    # 4. Build a fresh factory from the reduced inferred + base edges and train RotatE.
    ho_triples = []
    for f, n in highin:      ho_triples.append((f, 'highIn', n))
    for n, d in kept_treats: ho_triples.append((n, 'treats', d))
    for n, d in worsens:     ho_triples.append((n, 'worsensWith', d))
    for f, d in rec_train:   ho_triples.append((f, 'recommendedFor', d))
    for f, d in contra:      ho_triples.append((f, 'contraindicatedFor', d))

    ho_tf = TriplesFactory.from_labeled_triples(_np.array(ho_triples))
    ho_train, ho_valid, ho_test = ho_tf.split(ratios=[0.8, 0.1, 0.1], random_state=42)

    ho_result = pipeline(
        model='RotatE',
        training=ho_train, validation=ho_valid, testing=ho_test,
        model_kwargs=dict(embedding_dim=64),
        training_kwargs=dict(num_epochs=100, batch_size=256),
        random_seed=42,
    )
    ho_model = ho_result.model

    # 5. Can RotatE surface the held-out foods it was never trained to recommend?
    ho_food_set = set(f for f, _ in highin)
    K = 20
    hits = 0
    total = 0
    disease_hit = 0
    for d, target_foods in held_by_disease.items():
        try:
            pdf = predict_target(model=ho_model, head=None, relation='recommendedFor',
                                 tail=d, triples_factory=ho_tf).df
        except Exception:
            continue
        pdf = pdf[pdf['head_label'].isin(ho_food_set)]
        topk = set(pdf['head_label'].head(K))
        # Exclude foods that are STILL rule-derivable for d (we only score genuine held-outs).
        genuine = {f for f in target_foods if (f, d) not in rec_train}
        if not genuine:
            continue
        h = len(topk & genuine)
        hits += h
        total += len(genuine)
        if h > 0:
            disease_hit += 1

    hit_rate = hits / total if total else float('nan')
    print(f"\nHeld-out generalization @top-{K}:")
    print(f"  held-out (food,disease) pairs scored: {total}")
    print(f"  pairs recovered in top-{K}: {hits}  -> hit rate = {hit_rate:.3f}")
    print(f"  diseases with >=1 recovered held-out food: {disease_hit}/{len(held_by_disease)}")
    print("  (A positive hit rate = RotatE predicting food-disease links the reduced rule "
          "cannot derive, i.e. generalization beyond the symbolic rule.)")
else:
    print("Held-out generalization experiment skipped (RUN_HELDOUT = False).")


Held out 15 of 200 treats edges.
Held-out recommendedFor (food,disease) pairs the reduced rule cannot derive: 908 across 14 diseases.


INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [14423, 1865, 1866]


INFO:pykeen.pipeline.api:Using device: None


INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


...

Training batches on cpu:   0%|          | 0.00/59.0 [00:00<?, ?batch/s]

Training batches on cpu:   0%|          | 0.00/59.0 [00:00<?, ?batch/s]

Evaluating on cpu:   0%|          | 0.00/1.87k [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 1.51s seconds



Held-out generalization @top-20:
  held-out (food,disease) pairs scored: 259
  pairs recovered in top-20: 22  -> hit rate = 0.085
  diseases with >=1 recovered held-out food: 2/14
  (A positive hit rate = RotatE predicting food-disease links the reduced rule cannot derive, i.e. generalization beyond the symbolic rule.)


## 9. Visualization

The interactive graph visualization is provided in `outputs/knowledge_graph.html`.
It uses a D3.js force-directed layout with a sample of representative triples from the KG.

**Node types:** Food (orange), Nutrient (blue), Disease (purple)
**Edge types:** `highIn` (yellow), `treats` (green), `worsensWith` (orange),
`recommendedFor` (green), `contraindicatedFor` (red), `predictedFor` (teal)

Open `outputs/knowledge_graph.html` in any modern browser to explore the graph interactively:
- Click any node to inspect its relationships in the sidebar
- Drag nodes to rearrange the layout
- Scroll to zoom in/out

The full KG triples are available in `outputs/triples.csv` and `outputs/nutrition_kg_final.ttl`.


## 10. Limitations and Learning Outcome Alignment

### Limitations

This section honestly documents the simplifications that remain in this course project.

#### Data and Modeling Limitations

1. **Nutrient discretization** — only foods above the top quartile threshold are flagged as
   `highIn` a nutrient. This binary cutoff ignores the continuous, dose-dependent nature of
   nutrition: a food just above the 75th percentile and one far above it are treated
   identically. The quartile threshold is a pragmatic simplification.

2. **CTD mapping coverage** — only 8 of 18 tracked nutrients were matched to CTD therapeutic
   associations via exact name lookup. Vitamins C, B6, B12, Folate, and macronutrients
   (Protein, Fat, Carbohydrate) lack mappings, limiting recommendation coverage for diseases
   where these nutrients are clinically relevant (e.g., Vitamin C deficiency / Scurvy).

3. **Curated `worsensWith` coverage** — `worsensWith` associations are sourced from a
   hand-curated CSV (`worsensWith_curated.csv`) derived from clinical dietary guidelines
   (NIH ODS, WHO, AHA, ADA, EASL). The CTD `marker/mechanism` evidence class was evaluated
   but discarded because it conflates diagnostic biomarker associations (a nutrient measured
   as a disease marker in blood tests) with dietary harm (excess nutrient intake causing or
   aggravating a disease). For example, CTD marks Magnesium as a `marker/mechanism` for
   Magnesium Deficiency because serum magnesium is used to diagnose the condition — not
   because dietary magnesium intake worsens it. The curated approach trades recall for
   precision: fewer disease-nutrient pairs overall, but each pair is clinically meaningful
   as a dietary avoidance recommendation. A production system would use a structured
   vocabulary such as NDF-RT or DrugBank Diet Interactions with evidence grading.

4. **No food categorization hierarchy** — foods are individual USDA items with no
   taxonomic structure (no "legumes → lentils" hierarchy). An ontology hierarchy would
   enable principled generalization ("if legumes are recommended, suggest lentils, chickpeas...").

5. **Disease name normalization** — disease names with punctuation are converted to URI-safe
   strings via character substitution. A production system would require proper IRI encoding
   (RFC 3987) and MeSH/SNOMED alignment.

#### Reasoning Limitations

6. **Non-recursive rules only** — SPARQL CONSTRUCT handles the two single-hop rules
   correctly. Recursive reasoning (e.g., disease co-morbidity chains, transitive nutrient
   interactions) is not supported without a full Datalog engine.

7. **Contradiction resolution is destructive** — removing `recommendedFor` triples when
   a contraindication exists is a simple but lossy strategy. A more principled approach
   would retain provenance (why both were inferred) and let the scoring function arbitrate
   based on quantified risk.

#### Embedding Limitations

8. **Small KG** — ~520 entities is small by KGE standards. Embedding quality is inherently
   limited; absolute metrics reflect structure on a small graph, not SOTA-comparable performance.

9. **The embedding approximates the symbolic rules; it does not replace them.** RotatE is
   trained on **all five** relations, including both inferred food–disease relations
   (`recommendedFor`, `contraindicatedFor`), split 80/10/10. This is a correction of an earlier
   design in which those two relations were added to the vocabulary only and never trained — so
   their relation embeddings stayed at random initialisation and `predict_target()` for them
   ranked foods through a random rotation. The earlier write-up asymmetrically attributed
   "geometric signal" to `recommendedFor` but not `contraindicatedFor`; in fact **both were
   equally untrained**, and the "hazelnuts in both lists for Hypertension" contradiction was a
   symptom of that, not of any intrinsic difference between the relations. With both relations
   now trained, the per-relation evaluation (Section 7) and the verification cells show that
   RotatE recovers the deterministic rule-derived `recommendedFor` set well above random/degree
   baselines (overlap@10 = 1.00, vs 0.74 degree / 0.25 random baseline) and reaches filtered
   Hits@10 = 0.99 / MRR = 0.98 on its own test triples — i.e. the embedding is a soft,
   generalising approximation of the symbolic rules. `contraindicatedFor` is likewise trained now
   and ranks well (Hits@10 = 0.99, MRR = 0.94). The held-out experiment additionally shows RotatE
   recovering food–disease links whose generating `treats` edge was removed before reasoning
   (hit rate = 0.085 at top-20: 22 of 259 held-out pairs, 2 of 14 diseases), modest but positive
   evidence of generalization beyond a single rule instance. `predictedFor` is redefined in the section 8.4
   experiment as embedding-*completed* recommendations: because `recommendedFor` is saturated,
   the embedding cannot extend it, so we instead predict the genuinely-lossy `highIn` relation
   (a hard 75th-percentile cut on a continuous amount), validate those predictions against a
   softer 60-75th-percentile near-miss band (lift reported in 8.4), add the accepted edges to a
   graph *copy*, and propagate them through Rule 1. `predictedFor(F,D)` is a recommendation
   derivable only after the predicted `highIn` edges are added - never from observed data. This
   is approximate completion validated against a softer threshold, not against ground truth.
   Avoidance is nonetheless still **served** by the logical rules: they are exact and complete for
   the single-hop contraindication pattern, and the advisor surfaces the §8.4
   `predictedFor` edges, which are built disjoint from rule `recommendedFor` and exclude
   rule-contraindicated foods, so the served recommend/predict/avoid lists can never contradict (verified).
   This is a deliberate neuro-symbolic division of labour — symbolic rules for guaranteed
   correctness, embeddings for inductive extension — not a claim that one relation lacks signal.

10. **No hyperparameter optimization or model comparison** — RotatE is used with default
    settings (`embedding_dim=64`, `num_epochs=100`, `random_seed=42`). A rigorous study would
    compare TransE, ComplEx, and RotatE, and use cross-validated hyperparameter search.

---

### Learning Outcome Alignment

| LO | Level | Evidence |
|---|---|---|
| **LO1** — KG Embeddings | Strong | RotatE trained via PyKEEN on all five relations; per-relation MRR/Hits@10 reported (filtered) with caveats; link prediction in `dietary_advice` extends recommendations via `predictedFor` triples; embedding fix verified empirically (relations trained, embeddings recover the rule set above baseline, per-relation ranking beats random) |
| **LO2** — Logical Knowledge | Strong | Datalog notation used as formal spec; SPARQL CONSTRUCT as execution; distinction between Datalog and SPARQL explained; contradiction resolution |
| **LO4** — Data Models | Moderate | RDF triple model throughout; namespace design; RDFS schema (domain/range/labels); property graph contrast implicit in architecture discussion |
| **LO5** — KG Architecture | Moderate | Multi-layer pipeline: ingestion → construction → reasoning → embeddings → service; HTML visualization; SPARQL query interface |
| **LO6** — Scalable Reasoning | Basic | Single-pass SPARQL CONSTRUCT reasoning demonstrated; limitations of non-recursive SPARQL discussed; owlrl noted as extension point |
| **LO7** — KG Construction | Strong | Three-source integration: USDA FoodData Central (`highIn`, top-quartile threshold), CTD therapeutic evidence (`treats`), curated NIH/WHO/AHA/ADA guidelines (`worsensWith`); URI normalization; namespace design; RDF serialization to Turtle |
| **LO8** — KG Evolution | Strong | Rule-based inference derives 18k+ triples; RotatE *completes* the thresholded `highIn` relation and the recovered edges propagate through Rule 1 to add `predictedFor` triples (§8.4); contradiction handling removes conflicting triples |
| **LO9** — Real-world Applications | Strong | Healthcare/nutrition domain; explainable ranked recommendations; 150+ diseases covered; clinically grounded contraindications |
| **LO11** — KG Services | Moderate | `dietary_advice` service surfaces the propagation-derived `predictedFor` edges (the neuro-symbolic loop reaches the user-facing layer) with explainable nutrient bridges; structured output; HTML visualization; SPARQL SELECT queries; CSV export for downstream use |
| **LO12** — KGs, ML and AI | Strong | Explicit neuro-symbolic integration (rules + embeddings); both inferred relations trained; embeddings recover and generalize the symbolic rules (verified, held-out experiment); rules retained for guaranteed-correct avoidance; served lists provably non-contradictory; trade-off discussed honestly |
